<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## Week 7 · Session 2 · SQL Joins, Validation, and a Mini Data Project

        **Course level:** absolute beginner  
        **Prior learning:** SQLAlchemy engines, tables, CRUD, keys, and transactions from Session 7.1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        The final session links tables through keys. Students learn what each join preserves, how one-to-many relationships change row counts, and how to audit unmatched records before presenting a grouped result.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - predict the rows preserved by inner, left, and right joins
- build joins with SQLAlchemy Core
- identify unmatched keys on either side
- explain how key cardinality affects result size
- complete an end-to-end database summary with validation checks


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Recall | 10 min | Retrieve earlier ideas with a small runnable example |
| Learn | 30 min | Add new concepts in short, explained steps |
| Guided lab | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation first, predict the output, then run the code with <code>Shift + Enter</code>. Only edit cells marked <strong>Student task</strong>. Ask what every line contributes.</div>


### Check SQLAlchemy again

Every notebook is self-contained. The guarded examples give installation guidance instead of crashing when the optional package is absent.


In [ ]:
from importlib.util import find_spec

SQLALCHEMY_AVAILABLE = find_spec("sqlalchemy") is not None
print("SQLAlchemy available:", SQLALCHEMY_AVAILABLE)

if SQLALCHEMY_AVAILABLE:
    from sqlalchemy import Column, Float, ForeignKey, Integer, MetaData, String, Table
    from sqlalchemy import create_engine, func, insert, select
else:
    print("Install with: conda install sqlalchemy")


## 1. Create two related tables

The relationship is **one customer to many orders**:

- `customers.customer_id` is a primary key and therefore unique;
- `orders.customer_id` is a foreign key and may repeat;
- one order may have no matching customer in imperfect imported data.

For teaching unmatched keys, SQLite foreign-key enforcement is not enabled in this in-memory example. A production database should enforce valid relationships whenever the business rules require them.


### Build and populate the practice database

Customer 4 has no order, and order 105 refers to missing customer 99. Those cases reveal what each join preserves.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    engine = create_engine("sqlite+pysqlite:///:memory:")
    metadata = MetaData()
    customers_table = Table(
        "customers",
        metadata,
        Column("customer_id", Integer, primary_key=True),
        Column("customer_name", String(80), nullable=False),
        Column("region", String(30), nullable=False),
    )
    orders_table = Table(
        "orders",
        metadata,
        Column("order_id", Integer, primary_key=True),
        Column("customer_id", Integer, ForeignKey("customers.customer_id")),
        Column("amount", Float, nullable=False),
    )
    metadata.create_all(engine)

    with engine.begin() as connection:
        connection.execute(insert(customers_table), [
            {"customer_id": 1, "customer_name": "Asha", "region": "North"},
            {"customer_id": 2, "customer_name": "Bilal", "region": "South"},
            {"customer_id": 3, "customer_name": "Cara", "region": "North"},
            {"customer_id": 4, "customer_name": "Diego", "region": "West"},
        ])
        connection.execute(insert(orders_table), [
            {"order_id": 101, "customer_id": 1, "amount": 35.0},
            {"order_id": 102, "customer_id": 1, "amount": 20.0},
            {"order_id": 103, "customer_id": 2, "amount": 42.5},
            {"order_id": 104, "customer_id": 3, "amount": 18.0},
            {"order_id": 105, "customer_id": 99, "amount": 27.0},
        ])
    print("Customers and orders are ready.")
else:
    print("Database setup skipped.")


### Inspect both source tables

Always understand the inputs before joining them. Compare key values and row counts manually.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    with engine.connect() as connection:
        customer_rows = connection.execute(
            select(customers_table).order_by(customers_table.c.customer_id)
        ).mappings().all()
        order_rows = connection.execute(
            select(orders_table).order_by(orders_table.c.order_id)
        ).mappings().all()
    print("Customers:", [dict(row) for row in customer_rows])
    print("Orders:", [dict(row) for row in order_rows])
else:
    print("Source inspection skipped.")


## 2. Predict join behaviour

| Join | Rows preserved | Unmatched values |
|---|---|---|
| Inner | Only matching keys from both sides | removed |
| Left | Every row from the left table | right-side columns become null |
| Right | Every row from the right table | left-side columns become null |

SQLite does not provide a portable right-join workflow across all versions. We obtain the same preservation rule by swapping table order and performing a left outer join.


### Inner join: matched customers and orders

Customer 1 appears twice because one customer has two matching orders. Unmatched customer 4 and order 105 disappear.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    inner_statement = (
        select(
            customers_table.c.customer_id,
            customers_table.c.customer_name,
            orders_table.c.order_id,
            orders_table.c.amount,
        )
        .select_from(
            customers_table.join(
                orders_table,
                customers_table.c.customer_id == orders_table.c.customer_id,
            )
        )
        .order_by(orders_table.c.order_id)
    )
    with engine.connect() as connection:
        inner_rows = connection.execute(inner_statement).mappings().all()
    print("Inner rows:", len(inner_rows))
    print([dict(row) for row in inner_rows])
else:
    print("Inner join skipped.")


### Left join: preserve every customer

Customer 4 remains, but its order columns are `None` because there is no match.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    left_statement = (
        select(
            customers_table.c.customer_id,
            customers_table.c.customer_name,
            orders_table.c.order_id,
            orders_table.c.amount,
        )
        .select_from(
            customers_table.outerjoin(
                orders_table,
                customers_table.c.customer_id == orders_table.c.customer_id,
            )
        )
        .order_by(customers_table.c.customer_id, orders_table.c.order_id)
    )
    with engine.connect() as connection:
        left_rows = connection.execute(left_statement).mappings().all()
    print("Left rows:", len(left_rows))
    print([dict(row) for row in left_rows])
else:
    print("Left join skipped.")


### Right-join result by swapping table order

Orders are placed on the left of a left outer join, so every order is preserved. Order 105 has null customer details.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    preserve_orders_statement = (
        select(
            orders_table.c.order_id,
            orders_table.c.customer_id,
            customers_table.c.customer_name,
            orders_table.c.amount,
        )
        .select_from(
            orders_table.outerjoin(
                customers_table,
                orders_table.c.customer_id == customers_table.c.customer_id,
            )
        )
        .order_by(orders_table.c.order_id)
    )
    with engine.connect() as connection:
        preserve_order_rows = connection.execute(
            preserve_orders_statement
        ).mappings().all()
    print([dict(row) for row in preserve_order_rows])
else:
    print("Preserve-orders join skipped.")


## 3. Audit unmatched keys explicitly

Do not rely only on visually scanning a join. Add a null condition to turn unmatched records into a clear exception report.


### Find customers without orders

After a customer-preserving outer join, a null order primary key identifies a customer with no match.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    customers_without_orders = (
        select(customers_table.c.customer_id, customers_table.c.customer_name)
        .select_from(
            customers_table.outerjoin(
                orders_table,
                customers_table.c.customer_id == orders_table.c.customer_id,
            )
        )
        .where(orders_table.c.order_id.is_(None))
    )
    with engine.connect() as connection:
        no_order_rows = connection.execute(customers_without_orders).mappings().all()
    print([dict(row) for row in no_order_rows])
else:
    print("Customer audit skipped.")


### Find orders without customers

Swap the preservation side, then look for a null customer primary key.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    orders_without_customers = (
        select(orders_table.c.order_id, orders_table.c.customer_id)
        .select_from(
            orders_table.outerjoin(
                customers_table,
                orders_table.c.customer_id == customers_table.c.customer_id,
            )
        )
        .where(customers_table.c.customer_id.is_(None))
    )
    with engine.connect() as connection:
        orphan_order_rows = connection.execute(orders_without_customers).mappings().all()
    print([dict(row) for row in orphan_order_rows])
else:
    print("Order audit skipped.")


## 4. Cardinality controls result size

**Cardinality** describes how keys relate:

- one-to-one: each key appears at most once on each side;
- one-to-many: a unique key on one side can match several rows on the other;
- many-to-many: repeated keys on both sides can multiply rows unexpectedly.

Check key uniqueness before a join and compare row counts afterward.


### Measure key repetition

The query counts orders for every customer key. Customer 1 has two matches, so its joined row appears twice.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    orders_per_customer = (
        select(
            orders_table.c.customer_id,
            func.count(orders_table.c.order_id).label("order_count"),
        )
        .group_by(orders_table.c.customer_id)
        .order_by(orders_table.c.customer_id)
    )
    with engine.connect() as connection:
        cardinality_rows = connection.execute(orders_per_customer).mappings().all()
    print([dict(row) for row in cardinality_rows])
else:
    print("Cardinality check skipped.")


## 5. Join, group, and aggregate

Once the relationship is checked, the database can calculate a business summary. Use a left join when zero-activity customers or regions must remain visible.


### Summarise customer activity

`count(order_id)` ignores null order IDs, while `coalesce(sum(amount), 0)` converts a missing total to zero for customers without orders.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    customer_summary_statement = (
        select(
            customers_table.c.customer_id,
            customers_table.c.customer_name,
            func.count(orders_table.c.order_id).label("order_count"),
            func.coalesce(func.sum(orders_table.c.amount), 0).label("total_amount"),
        )
        .select_from(
            customers_table.outerjoin(
                orders_table,
                customers_table.c.customer_id == orders_table.c.customer_id,
            )
        )
        .group_by(customers_table.c.customer_id, customers_table.c.customer_name)
        .order_by(func.coalesce(func.sum(orders_table.c.amount), 0).desc())
    )
    with engine.connect() as connection:
        customer_summary = connection.execute(
            customer_summary_statement
        ).mappings().all()
    print([dict(row) for row in customer_summary])
else:
    print("Grouped join skipped.")


### Produce a region-level report

The join supplies each order's region, and grouping produces one result per region. Regions with no orders remain because customers are preserved.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    region_report_statement = (
        select(
            customers_table.c.region,
            func.count(func.distinct(customers_table.c.customer_id)).label("customers"),
            func.count(orders_table.c.order_id).label("orders"),
            func.coalesce(func.sum(orders_table.c.amount), 0).label("revenue"),
        )
        .select_from(
            customers_table.outerjoin(
                orders_table,
                customers_table.c.customer_id == orders_table.c.customer_id,
            )
        )
        .group_by(customers_table.c.region)
        .order_by(func.coalesce(func.sum(orders_table.c.amount), 0).desc())
    )
    with engine.connect() as connection:
        region_report = connection.execute(region_report_statement).mappings().all()
    print([dict(row) for row in region_report])
else:
    print("Region report skipped.")


## Mini-project checklist

Before presenting a joined result, record:

1. the question and required grain—one row per what?
2. the primary and foreign keys;
3. source row counts and key uniqueness;
4. which table must be preserved;
5. unmatched-key counts;
6. joined row count and duplicate explanation;
7. one manual spot-check against source rows.


## Student coding lab setup

The final five tasks use authors and books. Author 4 has no book; book 15 has no matching author. Run the setup cell once, then complete each task.


### Create the authors and books tables

This independent database can be reset by rerunning this cell.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    lab_engine = create_engine("sqlite+pysqlite:///:memory:")
    lab_metadata = MetaData()
    authors_table = Table(
        "authors",
        lab_metadata,
        Column("author_id", Integer, primary_key=True),
        Column("author_name", String(80), nullable=False),
        Column("country", String(40), nullable=False),
    )
    books_table = Table(
        "books",
        lab_metadata,
        Column("book_id", Integer, primary_key=True),
        Column("author_id", Integer, ForeignKey("authors.author_id")),
        Column("title", String(120), nullable=False),
        Column("sales", Integer, nullable=False),
    )
    lab_metadata.create_all(lab_engine)
    with lab_engine.begin() as connection:
        connection.execute(insert(authors_table), [
            {"author_id": 1, "author_name": "Aria", "country": "UK"},
            {"author_id": 2, "author_name": "Basil", "country": "Egypt"},
            {"author_id": 3, "author_name": "Cleo", "country": "UK"},
            {"author_id": 4, "author_name": "Dani", "country": "Spain"},
        ])
        connection.execute(insert(books_table), [
            {"book_id": 11, "author_id": 1, "title": "Data Paths", "sales": 120},
            {"book_id": 12, "author_id": 1, "title": "Clean Tables", "sales": 85},
            {"book_id": 13, "author_id": 2, "title": "Python Steps", "sales": 140},
            {"book_id": 14, "author_id": 3, "title": "Array Stories", "sales": 70},
            {"book_id": 15, "author_id": 99, "title": "Unknown Link", "sales": 20},
        ])
    print("Final lab ready.")
else:
    print("Install SQLAlchemy before completing the lab.")


        ### Student task 1: Predict before joining

        <div class="task"><strong>Your job:</strong> Write predictions for the inner-join row count, author-preserving left-join row count, number of authors without books, and number of books without authors. Then confirm source row counts in code.</div>

        **Check your work**

        - All four predictions are recorded before querying joins.
- Both source tables are counted.
- Predictions are later compared with evidence.


In [ ]:
# STUDENT TASK 1
predicted_inner_rows = 0
predicted_left_rows = 0
predicted_authors_without_books = 0
predicted_books_without_authors = 0

print("Predictions:", predicted_inner_rows, predicted_left_rows)
if SQLALCHEMY_AVAILABLE:
    with lab_engine.connect() as connection:
        author_count = len(connection.execute(select(authors_table)).all())
        book_count = len(connection.execute(select(books_table)).all())
    print("Source counts:", author_count, book_count)
else:
    print("Count check skipped.")


        ### Student task 2: Build an inner join

        <div class="task"><strong>Your job:</strong> Return author name, book title, and sales for matched keys only. Order from highest to lowest sales and print the row count.</div>

        **Check your work**

        - Only matching rows are returned.
- All three requested columns appear.
- The largest sales value is first.


In [ ]:
# STUDENT TASK 2
if SQLALCHEMY_AVAILABLE:
    author_book_inner = select(authors_table.c.author_name)
    # Replace with the requested columns, join, and order.
    with lab_engine.connect() as connection:
        matched_books = connection.execute(author_book_inner).mappings().all()
    print("Rows:", len(matched_books))
    print([dict(row) for row in matched_books])
else:
    print("Inner-join task skipped.")


        ### Student task 3: Preserve every author

        <div class="task"><strong>Your job:</strong> Create a left outer join beginning with authors. Return author name, country, book title, and sales, including authors with no books.</div>

        **Check your work**

        - Every author is present.
- The author without a book has null book values.
- The one-to-many author appears once per matching book.


In [ ]:
# STUDENT TASK 3
if SQLALCHEMY_AVAILABLE:
    all_authors_statement = select(authors_table)
    # Replace with the requested columns and outer join.
    with lab_engine.connect() as connection:
        all_authors_rows = connection.execute(all_authors_statement).mappings().all()
    print([dict(row) for row in all_authors_rows])
else:
    print("Left-join task skipped.")


        ### Student task 4: Produce two unmatched-key reports

        <div class="task"><strong>Your job:</strong> Query authors without books and books without authors. Print both counts and identifying columns.</div>

        **Check your work**

        - Each preservation side is correct.
- Null checks use primary-key columns from the non-preserved table.
- Both exception counts match the data.


In [ ]:
# STUDENT TASK 4
if SQLALCHEMY_AVAILABLE:
    authors_without_books_statement = select(authors_table).where(authors_table.c.author_id < 0)
    books_without_authors_statement = select(books_table).where(books_table.c.book_id < 0)
    # Replace both statements with outer joins and null checks.
    with lab_engine.connect() as connection:
        authors_without_books_rows = connection.execute(
            authors_without_books_statement
        ).mappings().all()
        books_without_authors_rows = connection.execute(
            books_without_authors_statement
        ).mappings().all()
    print("Authors without books:", [dict(row) for row in authors_without_books_rows])
    print("Books without authors:", [dict(row) for row in books_without_authors_rows])
else:
    print("Unmatched-key task skipped.")


        ### Student task 5: Complete the final country report

        <div class="task"><strong>Your job:</strong> Preserve all authors and produce one row per country with distinct author count, matched book count, and total sales. Convert missing sales to zero and sort by total sales descending.</div>

        **Check your work**

        - There is one row per country.
- Authors, books, and sales have meaningful labels.
- Spain remains with zero books and sales; rows are correctly sorted.



**Optional extension:** Write a five-line validation note comparing predictions, source counts, unmatched keys, result grain, and one manual spot-check.


In [ ]:
# STUDENT TASK 5
if SQLALCHEMY_AVAILABLE:
    country_report_statement = select(authors_table.c.country)
    # Replace with join, labelled aggregations, group, and order.
    with lab_engine.connect() as connection:
        country_report_rows = connection.execute(
            country_report_statement
        ).mappings().all()
    print([dict(row) for row in country_report_rows])
else:
    print("Country-report task skipped.")


        ## Knowledge check

        Answer these without running new code first.

        1. Which rows does an inner join remove?
2. How can a left join reproduce right-join preservation?
3. Why can one source row appear several times after a valid join?
4. How do you query unmatched rows explicitly?
5. What checks should accompany a published joined summary?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Revisit your Week 1 data-science question and describe how Python, NumPy, pandas, and SQL could support one reproducible answer. Keep practising with small, explainable datasets before increasing scale.
